# 01 - Data Cleaning et Analyse Exploratoire

Objectif : nettoyer le dataset de churn telecom, comprendre la structure des donnees et identifier les premiers facteurs associes au churn.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", None)

## Chargement des donnees

In [2]:
df = pd.read_csv("../dataset.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print("Shape:", df.shape)
df.info()

Shape: (7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 n

## Qualite des donnees

In [6]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
print("Valeurs manquantes visibles:")
print(missing)
print("\nDoublons:", duplicates)


Valeurs manquantes visibles:
customerID          0
DeviceProtection    0
TotalCharges        0
MonthlyCharges      0
PaymentMethod       0
PaperlessBilling    0
Contract            0
StreamingMovies     0
StreamingTV         0
TechSupport         0
OnlineBackup        0
gender              0
OnlineSecurity      0
InternetService     0
MultipleLines       0
PhoneService        0
tenure              0
Dependents          0
Partner             0
SeniorCitizen       0
Churn               0
dtype: int64

Doublons: 0


La colonne `TotalCharges` est lue comme texte. Dans ce dataset, certaines valeurs sont des espaces vides, ce qui bloque la conversion numerique.

In [7]:
blank_total = df["TotalCharges"].astype(str).str.strip().eq("")
print("Lignes avec TotalCharges vide:", blank_total.sum())
df.loc[blank_total, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Lignes avec TotalCharges vide: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


## Nettoyage

In [8]:
df_clean = df.copy()
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
df_clean["TotalCharges"] = df_clean["TotalCharges"].fillna(0)
df_clean["ChurnBinary"] = df_clean["Churn"].map({"No": 0, "Yes": 1})

df_clean.to_csv("../data/cleaned_dataset.csv", index=False)
df_clean.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBinary
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


## Distribution de la cible

In [9]:
churn_counts = df_clean["Churn"].value_counts()
churn_rates = df_clean["Churn"].value_counts(normalize=True).mul(100).round(2)
print(churn_counts)
print("\nPourcentages:")
print(churn_rates)


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Pourcentages:
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


## Analyse des variables numeriques

In [11]:
df_clean[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7043.00,7043.00
mean,0.16,32.37,64.76,2279.73
std,0.37,24.56,30.09,2266.79
min,0.00,0.00,18.25,0.00
25%,0.00,9.00,35.50,398.55
50%,0.00,29.00,70.35,1394.55
75%,0.00,55.00,89.85,3786.60
max,1.00,72.00,118.75,8684.80


## Taux de churn par variables categorielles

In [14]:
def churn_rate_by(column):
    return (
        df_clean.groupby(column)["ChurnBinary"]
        .mean()
        .mul(100)
        .sort_values(ascending=False)
        .round(1)
    )

for col in ["Contract", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity"]:
    print(f"\n{col}")
    print(churn_rate_by(col))

    print(churn_rate_by(col))


Contract
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: ChurnBinary, dtype: float64
Contract
Month-to-month    42.7
One year          11.3
Two year           2.8
Name: ChurnBinary, dtype: float64

PaymentMethod
PaymentMethod
Electronic check             45.3
Mailed check                 19.1
Bank transfer (automatic)    16.7
Credit card (automatic)      15.2
Name: ChurnBinary, dtype: float64
PaymentMethod
Electronic check             45.3
Mailed check                 19.1
Bank transfer (automatic)    16.7
Credit card (automatic)      15.2
Name: ChurnBinary, dtype: float64

InternetService
InternetService
Fiber optic    41.9
DSL            19.0
No              7.4
Name: ChurnBinary, dtype: float64
InternetService
Fiber optic    41.9
DSL            19.0
No              7.4
Name: ChurnBinary, dtype: float64

TechSupport
TechSupport
No                     41.6
Yes                    15.2
No internet service     7.4
Name: ChurnBinary, dtype: float64
Tec